# LAB | Ensemble Methods

Notebook resuelto en castellano siguiendo la estructura del lab y reutilizando el preprocesado del ejercicio anterior de **Feature Engineering**.

## 1. Cargar los datos

En este lab trabajamos con el mismo dataset **Spaceship Titanic** del ejercicio anterior.

Datos:
https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata:
https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

El objetivo es probar diferentes métodos de **ensemble learning** para comprobar si mejoran el modelo anterior. Para que la comparación sea justa, se mantiene el mismo proceso de limpieza, selección de variables, creación de dummies y escalado.

In [1]:
# Librerías principales
import pandas as pd
import numpy as np

# Preprocesado y división de datos
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Modelos ensemble
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import AdaBoostClassifier

# Métricas de evaluación
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [2]:
# Cargar el dataset
# Se usa la URL original del lab. El bloque except permite ejecutarlo también si tienes el CSV descargado en local.
data_url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv"

try:
    spaceship = pd.read_csv(data_url)
except Exception:
    spaceship = pd.read_csv("spaceship_titanic.csv")

spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


## 2. Preprocesado: Feature Engineering, Feature Selection y Scaling

Repetimos el mismo enfoque del lab anterior:

- Eliminamos filas con valores nulos, ya que el porcentaje de nulos es bajo.
- Transformamos `Cabin` para quedarnos solo con la cubierta, es decir, la primera letra.
- Eliminamos columnas demasiado granulares o identificativas, como `PassengerId` y `Name`.
- Convertimos variables categóricas en variables dummy.
- Separamos variable objetivo (`Transported`) y variables predictoras.

In [3]:
# Copia del dataframe original para trabajar sin modificar los datos brutos
spaceship_clean = spaceship.copy()

# Comprobación inicial de nulos
print("Shape original:", spaceship_clean.shape)
print("Valores nulos totales antes de limpiar:", spaceship_clean.isna().sum().sum())

# Eliminar filas con valores nulos
spaceship_clean = spaceship_clean.dropna().copy()

# Transformar Cabin: nos quedamos con la primera letra, que representa el deck/cubierta
spaceship_clean["Cabin"] = spaceship_clean["Cabin"].str[0]

# Definir variable objetivo
y = spaceship_clean["Transported"].astype(int)

# Seleccionar variables predictoras
# PassengerId y Name se eliminan porque son identificadores/texto demasiado granular
X = spaceship_clean.drop(columns=["Transported", "PassengerId", "Name"])

# Convertir variables categóricas y booleanas a variables dummy
X_encoded = pd.get_dummies(X, drop_first=True)

print("Shape tras limpieza:", spaceship_clean.shape)
print("Shape de X_encoded:", X_encoded.shape)

X_encoded.head()

Shape original: (8693, 14)
Valores nulos totales antes de limpiar: 2324
Shape tras limpieza: (6606, 14)
Shape de X_encoded: (6606, 19)


,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,HomePlanet_Europa,HomePlanet_Mars,CryoSleep_True,Cabin_B,Cabin_C,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,VIP_True
0,39.0,0.0,0.0,0.0,0.0,0.0,True,False,False,True,False,False,False,False,False,False,False,True,False
1,24.0,109.0,9.0,25.0,549.0,44.0,False,False,False,False,False,False,False,True,False,False,False,True,False
2,58.0,43.0,3576.0,0.0,6715.0,49.0,True,False,False,False,False,False,False,False,False,False,False,True,True
3,33.0,0.0,1283.0,371.0,3329.0,193.0,True,False,False,False,False,False,False,False,False,False,False,True,False
4,16.0,303.0,70.0,151.0,565.0,2.0,False,False,False,False,False,False,False,True,False,False,False,True,False


## 3. Train Test Split

Dividimos los datos en entrenamiento y test. Usamos `stratify=y` para mantener una proporción similar de clases en ambos conjuntos.

In [4]:
# División train/test
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Escalado de variables
# Aunque los modelos de árboles no lo necesitan estrictamente, lo mantenemos para respetar el preprocesado anterior.
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("X_train:", X_train_scaled.shape)
print("X_test:", X_test_scaled.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (5284, 19)
X_test: (1322, 19)
y_train: (5284,)
y_test: (1322,)


## 4. Model Selection

Ahora probamos varios métodos de ensemble para comparar su rendimiento:

- Bagging
- Pasting
- Random Forest
- Gradient Boosting
- AdaBoost

Para evaluar todos los modelos de forma consistente, usaremos `accuracy`, matriz de confusión y `classification_report`.

### 4.1. Bagging and Pasting

**Bagging** entrena varios modelos sobre muestras aleatorias con reemplazo.  
**Pasting** es parecido, pero usa muestras aleatorias sin reemplazo.

In [5]:
# Función auxiliar para entrenar y evaluar modelos

def evaluate_model(model, model_name, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)

    print(f"Modelo: {model_name}")
    print(f"Accuracy: {accuracy:.4f}")
    print("\nMatriz de confusión:")
    print(confusion_matrix(y_test, y_pred))
    print("\nClassification report:")
    print(classification_report(y_test, y_pred))

    return accuracy, y_pred


# Modelo base para Bagging/Pasting
base_tree = DecisionTreeClassifier(random_state=42)

# Bagging: muestreo con reemplazo
bagging_model = BaggingClassifier(
    estimator=base_tree,
    n_estimators=50,
    bootstrap=True,
    random_state=42,
    n_jobs=1
)

bagging_accuracy, bagging_pred = evaluate_model(
    bagging_model,
    "Bagging Classifier",
    X_train_scaled,
    X_test_scaled,
    y_train,
    y_test
)


# Pasting: muestreo sin reemplazo
pasting_model = BaggingClassifier(
    estimator=base_tree,
    n_estimators=50,
    bootstrap=False,
    random_state=42,
    n_jobs=1
)

pasting_accuracy, pasting_pred = evaluate_model(
    pasting_model,
    "Pasting Classifier",
    X_train_scaled,
    X_test_scaled,
    y_train,
    y_test
)

Modelo: Bagging Classifier
Accuracy: 0.7791

Matriz de confusión:
[[508 148]
 [144 522]]

Classification report:
              precision    recall  f1-score   support

           0       0.78      0.77      0.78       656
           1       0.78      0.78      0.78       666

    accuracy                           0.78      1322
   macro avg       0.78      0.78      0.78      1322
weighted avg       0.78      0.78      0.78      1322



Modelo: Pasting Classifier
Accuracy: 0.7390

Matriz de confusión:
[[454 202]
 [143 523]]

Classification report:
              precision    recall  f1-score   support

           0       0.76      0.69      0.72       656
           1       0.72      0.79      0.75       666

    accuracy                           0.74      1322
   macro avg       0.74      0.74      0.74      1322
weighted avg       0.74      0.74      0.74      1322



### 4.2. Random Forests

Random Forest combina muchos árboles de decisión. Además de usar muestras aleatorias, también selecciona subconjuntos aleatorios de variables en cada árbol, reduciendo el riesgo de overfitting.

In [6]:
# Random Forest
random_forest_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    random_state=42,
    n_jobs=1
)

rf_accuracy, rf_pred = evaluate_model(
    random_forest_model,
    "Random Forest",
    X_train_scaled,
    X_test_scaled,
    y_train,
    y_test
)

Modelo: Random Forest
Accuracy: 0.7799

Matriz de confusión:
[[511 145]
 [146 520]]

Classification report:
              precision    recall  f1-score   support

           0       0.78      0.78      0.78       656
           1       0.78      0.78      0.78       666

    accuracy                           0.78      1322
   macro avg       0.78      0.78      0.78      1322
weighted avg       0.78      0.78      0.78      1322



### 4.3. Gradient Boosting

Gradient Boosting entrena árboles de forma secuencial. Cada nuevo árbol intenta corregir los errores cometidos por los anteriores.

In [7]:
# Gradient Boosting
gradient_boosting_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

gb_accuracy, gb_pred = evaluate_model(
    gradient_boosting_model,
    "Gradient Boosting",
    X_train_scaled,
    X_test_scaled,
    y_train,
    y_test
)

Modelo: Gradient Boosting
Accuracy: 0.7837

Matriz de confusión:
[[471 185]
 [101 565]]

Classification report:
              precision    recall  f1-score   support

           0       0.82      0.72      0.77       656
           1       0.75      0.85      0.80       666

    accuracy                           0.78      1322
   macro avg       0.79      0.78      0.78      1322
weighted avg       0.79      0.78      0.78      1322



### 4.4. Adaptive Boosting

AdaBoost también entrena modelos de manera secuencial, dando más peso a las observaciones que se han clasificado mal en iteraciones anteriores.

In [8]:
# Adaptive Boosting
adaboost_model = AdaBoostClassifier(
    n_estimators=100,
    learning_rate=0.5,
    random_state=42
)

ada_accuracy, ada_pred = evaluate_model(
    adaboost_model,
    "AdaBoost",
    X_train_scaled,
    X_test_scaled,
    y_train,
    y_test
)

Modelo: AdaBoost
Accuracy: 0.7542

Matriz de confusión:
[[546 110]
 [215 451]]

Classification report:
              precision    recall  f1-score   support

           0       0.72      0.83      0.77       656
           1       0.80      0.68      0.74       666

    accuracy                           0.75      1322
   macro avg       0.76      0.75      0.75      1322
weighted avg       0.76      0.75      0.75      1322



## 5. ¿Qué modelo es el mejor y por qué?

Comparamos todos los modelos mediante la métrica `accuracy`. Además, revisamos el `classification_report` para no quedarnos solo con una métrica general.

In [9]:
# Comparación final de modelos
results = pd.DataFrame({
    "Modelo": [
        "Bagging",
        "Pasting",
        "Random Forest",
        "Gradient Boosting",
        "AdaBoost"
    ],
    "Accuracy": [
        bagging_accuracy,
        pasting_accuracy,
        rf_accuracy,
        gb_accuracy,
        ada_accuracy
    ]
}).sort_values(by="Accuracy", ascending=False)

display(results)

best_model_name = results.iloc[0]["Modelo"]
best_model_accuracy = results.iloc[0]["Accuracy"]

print(f"El mejor modelo según accuracy es {best_model_name}, con un accuracy de {best_model_accuracy:.4f}.")

print("""
Conclusión:
El mejor modelo es el que obtiene mayor accuracy en el conjunto de test. 
En este caso, los modelos basados en árboles funcionan bien porque el dataset combina variables numéricas, categóricas transformadas en dummies y relaciones no necesariamente lineales.

Random Forest suele ser una opción especialmente sólida porque reduce el overfitting de un único árbol de decisión y combina muchos árboles independientes.
Gradient Boosting y AdaBoost también pueden funcionar muy bien, pero son más sensibles a los hiperparámetros porque construyen modelos de forma secuencial.
""")

,Modelo,Accuracy
3,Gradient Boosting,0.783661
2,Random Forest,0.779879
0,Bagging,0.779123
4,AdaBoost,0.754160
1,Pasting,0.739032


El mejor modelo según accuracy es Gradient Boosting, con un accuracy de 0.7837.

Conclusión:
El mejor modelo es el que obtiene mayor accuracy en el conjunto de test. 
En este caso, los modelos basados en árboles funcionan bien porque el dataset combina variables numéricas, categóricas transformadas en dummies y relaciones no necesariamente lineales.

Random Forest suele ser una opción especialmente sólida porque reduce el overfitting de un único árbol de decisión y combina muchos árboles independientes.
Gradient Boosting y AdaBoost también pueden funcionar muy bien, pero son más sensibles a los hiperparámetros porque construyen modelos de forma secuencial.

